In [4]:

# ============================================================
# PHÂN LOẠI TIN NHẮN LỪA ĐẢO TRỰC TUYẾN
# TF-IDF + MULTINOMIAL NAIVE BAYES
# ============================================================

# 1. IMPORT THƯ VIỆN

import os
import json
import joblib
import pandas as pd

from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


# 2. KHAI BÁO CẤU HÌNH

TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"

TEXT_COLUMN = "text_clean"
LABEL_COLUMN = "label"

MODEL_PATH = "/content/naive_bayes_pipeline.pkl"
PREDICTION_PATH = "/content/y_test_y_pred.csv"
INFO_PATH = "/content/model_info.json"

EXPECTED_LABELS = {"ham", "spam", "smishing"}


# 3. ĐỌC DỮ LIỆU

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError("Không tìm thấy file train.csv")

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError("Không tìm thấy file test.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Kích thước tập Train:", train_df.shape)
print("Kích thước tập Test:", test_df.shape)


# 4. KIỂM TRA DỮ LIỆU

required_columns = [TEXT_COLUMN, LABEL_COLUMN]

for column in required_columns:
    if column not in train_df.columns or column not in test_df.columns:
        raise ValueError(f"Thiếu cột dữ liệu: {column}")

if train_df[required_columns].isnull().any().any():
    raise ValueError("Tập Train có dữ liệu bị thiếu.")

if test_df[required_columns].isnull().any().any():
    raise ValueError("Tập Test có dữ liệu bị thiếu.")

train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)
test_df[TEXT_COLUMN] = test_df[TEXT_COLUMN].astype(str)

if train_df[TEXT_COLUMN].str.strip().eq("").any():
    raise ValueError("Tập Train có nội dung tin nhắn rỗng.")

if test_df[TEXT_COLUMN].str.strip().eq("").any():
    raise ValueError("Tập Test có nội dung tin nhắn rỗng.")

if set(train_df[LABEL_COLUMN].unique()) != EXPECTED_LABELS:
    raise ValueError("Tập Train không có đủ ba nhãn hợp lệ.")

if set(test_df[LABEL_COLUMN].unique()) != EXPECTED_LABELS:
    raise ValueError("Tập Test không có đủ ba nhãn hợp lệ.")

print("\nPhân bố nhãn Train:")
print(train_df[LABEL_COLUMN].value_counts())

print("\nPhân bố nhãn Test:")
print(test_df[LABEL_COLUMN].value_counts())


# 5. XÁC ĐỊNH ĐẶC TRƯNG VÀ NHÃN

X_train = train_df[TEXT_COLUMN]
y_train = train_df[LABEL_COLUMN]

X_test = test_df[TEXT_COLUMN]
y_test = test_df[LABEL_COLUMN]

print("\nSố mẫu Train:", len(X_train))
print("Số mẫu Test:", len(X_test))


# 6. XÂY DỰNG PIPELINE TF-IDF + NAIVE BAYES

model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=50000,
            sublinear_tf=True
        )
    ),
    (
        "naive_bayes",
        MultinomialNB(alpha=1.0)
    )
])


# 7. HUẤN LUYỆN MÔ HÌNH

model.fit(X_train, y_train)

print("\nHuấn luyện mô hình hoàn tất!")


# 8. DỰ ĐOÁN TRÊN TẬP TEST

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("Dự đoán hoàn tất!")


# 9. ĐÁNH GIÁ ĐỘ CHÍNH XÁC

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(accuracy, 4))
print(f"Accuracy: {accuracy * 100:.2f}%")


# 10. LƯU KẾT QUẢ DỰ ĐOÁN

prediction_df = pd.DataFrame({
    "y_test": y_test.reset_index(drop=True),
    "y_pred": y_pred
})

if "id" in test_df.columns:
    prediction_df.insert(
        0,
        "id",
        test_df["id"].reset_index(drop=True)
    )

prediction_df["text_clean"] = X_test.reset_index(drop=True)

class_labels = model.named_steps["naive_bayes"].classes_

for index, label in enumerate(class_labels):
    prediction_df[f"probability_{label}"] = y_pred_proba[:, index]

prediction_df.to_csv(
    PREDICTION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu kết quả dự đoán:", PREDICTION_PATH)


# 11. LƯU PIPELINE MÔ HÌNH

joblib.dump(model, MODEL_PATH)

print("Đã lưu mô hình:", MODEL_PATH)


# 12. KIỂM TRA MÔ HÌNH ĐÃ LƯU

loaded_model = joblib.load(MODEL_PATH)

sample_message = X_test.iloc[0]

sample_prediction = loaded_model.predict([sample_message])[0]

print("\nTin nhắn kiểm tra:")
print(sample_message)

print("Nhãn dự đoán:", sample_prediction)


# 13. LƯU THÔNG TIN MÔ HÌNH

model_info = {
    "project": "Online Scam Message Classification",
    "algorithm": "Multinomial Naive Bayes",
    "vectorizer": "TF-IDF",
    "text_column": TEXT_COLUMN,
    "label_column": LABEL_COLUMN,
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "classes": [str(label) for label in class_labels],
    "accuracy": float(accuracy),
    "tfidf_parameters": {
        "lowercase": True,
        "ngram_range": [1, 2],
        "min_df": 2,
        "max_features": 50000,
        "sublinear_tf": True
    },
    "naive_bayes_parameters": {
        "alpha": 1.0
    },
    "created_at": datetime.now().isoformat()
}

with open(INFO_PATH, "w", encoding="utf-8") as file:
    json.dump(model_info, file, ensure_ascii=False, indent=4)


# 14. TỔNG KẾT

print("\n" + "=" * 50)
print("KẾT QUẢ HUẤN LUYỆN MÔ HÌNH")
print("=" * 50)

print("Thuật toán: Multinomial Naive Bayes")
print("Vectorizer: TF-IDF")
print("Số mẫu Train:", len(X_train))
print("Số mẫu Test:", len(X_test))
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nCác file kết quả:")

for file_path in [MODEL_PATH, PREDICTION_PATH, INFO_PATH]:
    print(
        f"{os.path.basename(file_path)}: "
        f"{'Đã tạo' if os.path.exists(file_path) else 'Chưa tạo'}"
    )

print("\nHoàn tất!")


Kích thước tập Train: (4637, 8)
Kích thước tập Test: (1160, 8)

Phân bố nhãn Train:
label
ham         3867
smishing     422
spam         348
Name: count, dtype: int64

Phân bố nhãn Test:
label
ham         967
smishing    106
spam         87
Name: count, dtype: int64

Số mẫu Train: 4637
Số mẫu Test: 1160

Huấn luyện mô hình hoàn tất!
Dự đoán hoàn tất!

Accuracy: 0.925
Accuracy: 92.50%

Đã lưu kết quả dự đoán: /content/y_test_y_pred.csv
Đã lưu mô hình: /content/naive_bayes_pipeline.pkl

Tin nhắn kiểm tra:
i am in escape theatre now. . going to watch kavalan in a few minutes
Nhãn dự đoán: ham

KẾT QUẢ HUẤN LUYỆN MÔ HÌNH
Thuật toán: Multinomial Naive Bayes
Vectorizer: TF-IDF
Số mẫu Train: 4637
Số mẫu Test: 1160
Accuracy: 92.50%

Các file kết quả:
naive_bayes_pipeline.pkl: Đã tạo
y_test_y_pred.csv: Đã tạo
model_info.json: Đã tạo

Hoàn tất!
